# ByT5 IPA G2P Training

Fine-tunes `google/byt5-small` on (headword → IPA) pairs for the LexCW dictionary.

**You need to upload `pairs.json` to this notebook's runtime first.**

1. Run Cell 1 to upload `pairs.json`
2. Run Cell 2 (installs deps)
3. Run Cell 3 (training — ~30 min on T4)
4. Run Cell 4 to download the model zip
5. Unzip into `instance/ipa_models/` on the server and restart

In [ ]:
from google.colab import files
print("Upload pairs.json from your machine...")
uploaded = files.upload()
!ls -lh pairs.json

In [ ]:
!pip install -q transformers datasets accelerate sentencepiece
print('Dependencies installed')

In [ ]:
import json, warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import torch
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)
from datasets import Dataset

# --- load pairs ---
with open('pairs.json') as f:
    raw = json.load(f)
pairs = [(d['headword'], d['ipa']) for d in raw if d.get('headword') and d.get('ipa')]
print(f'Loaded {len(pairs)} pairs')

sources = [h for h, _ in pairs]
targets = [i for _, i in pairs]

# --- model ---
model_name = 'google/byt5-small'
ipa_ws = 'seh-fonipa'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

# --- tokenize ---
def tok_fn(batch):
    src = tokenizer(batch['src'], truncation=True, max_length=128)
    tgt = tokenizer(batch['tgt'], truncation=True, max_length=128)
    labels = [[-100 if t == tokenizer.pad_token_id else t for t in ids] for ids in tgt['input_ids']]
    return {**src, 'labels': labels}

ds = Dataset.from_dict({'src': sources, 'tgt': targets}).map(tok_fn, batched=True, remove_columns=['src','tgt'])
split = ds.train_test_split(test_size=0.1, seed=42)
print(f'Train: {len(split["train"])}  Eval: {len(split["test"])}')

# --- training ---
# IMPORTANT: fp16=False -- with only ~130 training samples, fp16 produces NaN gradients
# and the model outputs garbage (@@@@@@). Train in fp32 for stability.
args = Seq2SeqTrainingArguments(
    output_dir='./byt5_checkpoints',
    eval_strategy='epoch',
    save_strategy='epoch',
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-4,
    num_train_epochs=100,
    fp16=False,
    predict_with_generate=True,
    generation_max_length=128,
    save_total_limit=2,
    logging_steps=10,
    report_to='none',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
)

collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)
trainer = Seq2SeqTrainer(
    model, args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    data_collator=collator,
)

trainer.train()

# --- save ---
out_dir = Path(f'./byt5_model/ipa_byt5_{ipa_ws}')
out_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(out_dir))
tokenizer.save_pretrained(str(out_dir))

with open(out_dir / 'metadata.json', 'w') as f:
    json.dump({'ipa_writing_system': ipa_ws, 'base_model': model_name,
               'source_prefix': '', 'task': 'g2p'}, f, indent=2)
print(f'Model saved to: {out_dir}')

# --- quick test ---
device = torch.device('cuda')
model.to(device)
model.eval()
test_words = ['acceptance test', 'acid test', 'attest', 'book test', 'crash test']
for w in test_words:
    inp = tokenizer(w, return_tensors='pt', truncation=True, max_length=128).to(device)
    with torch.no_grad():
        gen = model.generate(**inp, num_beams=4, max_length=128, early_stopping=True)
    pred = tokenizer.decode(gen[0], skip_special_tokens=True).strip()
    print(f'  {w:30s} -> {pred}')

In [ ]:
import shutil
from google.colab import files

model_dir = 'byt5_model/ipa_byt5_seh-fonipa'
!zip -r ipa_byt5_seh-fonipa.zip {model_dir}
files.download('ipa_byt5_seh-fonipa.zip')
print('Done. Unzip this into instance/ipa_models/ on the server and restart the app.')